# Data Description
- Background: Unit 1.7 補充資料驗證：Geofabrik OSM道路。提供機關：Geofabrik／OpenStreetMap contributors；官方來源：https://download.geofabrik.de/asia/taiwan.html。原始來源對應與官方最新資源bytes完全相同為不同命題；除歷史界內包已逐byte雜湊對應外，未重新下載認證其他最新資源。
- date received: 學生於本單元提供；實際下載／收到日期未提供。首次本機檢視2026-11-26，不將今日當下載日。
- Path to data file: 官方來源及版本如上；精確本機容器如下，解壓至data/temp僅為衍生暫存。
  - `data/raw/taiwan-250101-free.shp.zip`；215,526,345 bytes；SHA-256 `ad19e79129efc0a447db5fe5d684e732199a900251f04f2aacef821cfa9ff1ab`。
- Unit of observation: OSM道路線表示，可能分段及分離車道。
- Sample period: 包內README明示2025-01-01T21:21:00Z快照；非2024／2025逐年獨立路網；發布／更新：歷史資源檔名250101；精確發布時間未認證。ZIP時間、檔名時間、觀測／設置時間分開，不由檔名猜全年存量。
- Known issues: 僅讀道路及研究區外圍bbox；線表示密度非實體中心線、交通流量或VKT；免費格式欠完整OSM節點拓樸。缺失、重複、坐標及CRS由下方實讀輸出；沒有完整性證據的0僅為本檔未觀測。授權：Open Database License 1.0；需OpenStreetMap contributors標示及遵守ODbL。
- Definition for each variable: 下方逐欄型態、缺失、觀測域與定義表。已知官方欄意與本次診斷欄分開；未獲欄位碼表支持者明記未知，不依名稱猜設施單位。
- Data path: 上列每個獨立原始容器；實際讀取成員列於Summary Stats。
- Role: raw。
- Verification status: 以本Notebook乾淨kernel實際輸出與執行metadata為準；不認證已清理。
- SHA-256: 上列每檔固定綁定，載入前逐項重核。
- Next processing notebook: 本單元僅診斷，Chapter 2尚未建立；[驗證一](../code/taipei_supplementary_accident_boundary_validation.ipynb)、[驗證二](../code/taipei_transportation_infrastructure_validation.ipynb)。

Author: 詹明翰、謝子詮、曾翊涵；Created At / Last Modified At: 2026-11-26。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。共同作者與職務不代表已完成個別工作。本輪由Codex協助執行，成員驗收與審查尚待完成。


In [1]:
from pathlib import Path
import sys, hashlib, json, zipfile
import pandas as pd
from IPython.display import display, Markdown
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate/'AGENTS.md').is_file() and (candidate/'templates').is_dir():
        PROJECT_ROOT=candidate;break
else:raise FileNotFoundError('請從本研究工作包內執行')
def fingerprint(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_config.json')=='7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7'
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_support.py')=='1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
from unit17_validation_support import Context, show, safe_text
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',160)
ctx=Context(PROJECT_ROOT,'7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7')
print('Python:',sys.version,'Config:','7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7','Helper:','1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080')


Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)] Config: 7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7 Helper: 1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080


## Change for Your Own Data

使用已核對精確路徑及固定SHA-256；不選第一個glob或最新檔。CP950非法bytes只作可逆診斷；大檔分批讀取、OSM僅研究區2km bbox，保留全國母體數。

In [2]:
FAMILY='osm'
INPUTS=ctx.family_items(FAMILY)
for item in INPUTS:ctx.file(item['path'])
frames=ctx.load_family(FAMILY)

## Summary

每個容器、實際成員與載入範圍分列；沒有將暫存解壓檔當原始下載資料。

In [3]:
show('固定輸入身分',pd.DataFrame([{k:v for k,v in r.items() if k not in ['members','encoding_checks']} for r in INPUTS]))
show('讀取範圍與CRS／編碼',pd.DataFrame(ctx.summaries[FAMILY]))

**固定輸入身分**

,family,path,bytes,sha256,status
0,osm,data/raw/taiwan-250101-free.shp.zip,215526345,ad19e79129efc0a447db5fe5d684e732199a900251f04f2aacef821cfa9ff1ab,present


**讀取範圍與CRS／編碼**

,path,member,total_features,loaded_rows,crs,types,bounds,invalid,null_geometry,empty
0,data/raw/taiwan-250101-free.shp.zip,gis_osm_roads_free_1.shp,737817,93215,EPSG:4326,{'LineString': 93215},"[121.407863, 24.927588, 121.70125, 25.254497]",0,0,0


### Sample

各實際讀取表前十列；全國事故為臺北篩選後第一當事人，不代表全國前十列。OSM為bbox空間子集。

In [4]:
for source,frame in frames.items():
    show(source,frame.drop(columns='geometry',errors='ignore').head(10),block=10)

**data/raw/taiwan-250101-free.shp.zip::gis_osm_roads_free_1.shp**

,osm_id,code,fclass,name,ref,oneway,maxspeed,layer,bridge,tunnel,_source_row,_source_path,_source_member
fid,,,,,,,,,,,,,
0,4860359,5114,secondary,重慶北路一段,NaN,F,0,0,F,F,1,data/raw/taiwan-250101-free.shp.zip,gis_osm_roads_free_1.shp
30,4928571,5111,motorway,中山高速公路,1,F,100,1,T,F,31,data/raw/taiwan-250101-free.shp.zip,gis_osm_roads_free_1.shp
44,4947947,5122,residential,太原路,NaN,B,0,0,F,F,45,data/raw/taiwan-250101-free.shp.zip,gis_osm_roads_free_1.shp
45,4947948,5141,service,太原路115巷,NaN,F,0,0,F,F,46,data/raw/taiwan-250101-free.shp.zip,gis_osm_roads_free_1.shp
52,5117999,5114,secondary,承德路一段,NaN,F,0,0,F,F,53,data/raw/taiwan-250101-free.shp.zip,gis_osm_roads_free_1.shp
83,5909607,5131,motorway_link,NaN,NaN,F,60,0,F,F,84,data/raw/taiwan-250101-free.shp.zip,gis_osm_roads_free_1.shp
84,5909609,5131,motorway_link,NaN,NaN,F,0,2,T,F,85,data/raw/taiwan-250101-free.shp.zip,gis_osm_roads_free_1.shp
85,5909612,5131,motorway_link,NaN,NaN,F,0,1,T,F,86,data/raw/taiwan-250101-free.shp.zip,gis_osm_roads_free_1.shp
135,6183250,5135,tertiary_link,椰林大道,NaN,F,0,0,F,F,136,data/raw/taiwan-250101-free.shp.zip,gis_osm_roads_free_1.shp


### Summary Stats

ZIP逐成員CRC、配套、原始WKT、全部欄型態／缺失／定義、重複與數值／類別／時間診斷。保留原值及bytes。

In [5]:
_ = ctx.raw_report(FAMILY,crc=True)

,source_metadata_and_limits
title,Geofabrik OSM道路
provider,Geofabrik／OpenStreetMap contributors
url,https://download.geofabrik.de/asia/taiwan.html
publication,歷史資源檔名250101；精確發布時間未認證
period,包內README明示2025-01-01T21:21:00Z快照；非2024／2025逐年獨立路網
limitation,僅讀道路及研究區外圍bbox；線表示密度非實體中心線、交通流量或VKT；免費格式欠完整OSM節點拓樸
license,Open Database License 1.0；需OpenStreetMap contributors標示及遵守ODbL


,family,path,bytes,sha256,status
0,osm,data/raw/taiwan-250101-free.shp.zip,215526345,ad19e79129efc0a447db5fe5d684e732199a900251f04f2aacef821cfa9ff1ab,present


**ZIP原始成員清單；非解壓暫存資料來源**

,member,bytes,compressed,crc32,zip_datetime
0,README,652,370,e4c9f514,"[2025, 1, 2, 2, 55, 50]"
1,gis_osm_buildings_a_free_1.cpg,6,6,59b2d65b,"[2025, 1, 2, 2, 55, 20]"
2,gis_osm_buildings_a_free_1.dbf,79360574,2959775,a25aa1bd,"[2025, 1, 2, 2, 55, 50]"
3,gis_osm_buildings_a_free_1.prj,144,122,0bbee490,"[2025, 1, 2, 2, 55, 20]"
4,gis_osm_buildings_a_free_1.shp,80380524,37849952,614a9303,"[2025, 1, 2, 2, 55, 50]"
5,gis_osm_buildings_a_free_1.shx,3847876,1527234,7be93ab7,"[2025, 1, 2, 2, 55, 50]"
6,gis_osm_landuse_a_free_1.cpg,6,6,59b2d65b,"[2025, 1, 2, 2, 55, 20]"
7,gis_osm_landuse_a_free_1.dbf,14430852,737230,12b9d0d6,"[2025, 1, 2, 2, 55, 50]"
8,gis_osm_landuse_a_free_1.prj,144,122,0bbee490,"[2025, 1, 2, 2, 55, 20]"
9,gis_osm_landuse_a_free_1.shp,41808000,26466826,af3e9073,"[2025, 1, 2, 2, 55, 50]"


,member,bytes,compressed,crc32,zip_datetime
18,gis_osm_natural_free_1.prj,144,122,0bbee490,"[2025, 1, 2, 2, 55, 20]"
19,gis_osm_natural_free_1.shp,445972,252213,dc687170,"[2025, 1, 2, 2, 55, 50]"
20,gis_osm_natural_free_1.shx,127492,36529,c1e03c44,"[2025, 1, 2, 2, 55, 50]"
21,gis_osm_places_a_free_1.cpg,6,6,59b2d65b,"[2025, 1, 2, 2, 55, 20]"
22,gis_osm_places_a_free_1.dbf,12439,1389,6119906e,"[2025, 1, 2, 2, 55, 50]"
23,gis_osm_places_a_free_1.prj,144,122,0bbee490,"[2025, 1, 2, 2, 55, 20]"
24,gis_osm_places_a_free_1.shp,812020,571708,6d470c0e,"[2025, 1, 2, 2, 55, 50]"
25,gis_osm_places_a_free_1.shx,732,494,92be0339,"[2025, 1, 2, 2, 55, 50]"
26,gis_osm_places_free_1.cpg,6,6,59b2d65b,"[2025, 1, 2, 2, 55, 20]"
27,gis_osm_places_free_1.dbf,2626979,173225,9cccd865,"[2025, 1, 2, 2, 55, 50]"


,member,bytes,compressed,crc32,zip_datetime
36,gis_osm_pofw_free_1.cpg,6,6,59b2d65b,"[2025, 1, 2, 2, 55, 20]"
37,gis_osm_pofw_free_1.dbf,438352,43057,df45775c,"[2025, 1, 2, 2, 55, 50]"
38,gis_osm_pofw_free_1.prj,144,122,0bbee490,"[2025, 1, 2, 2, 55, 20]"
39,gis_osm_pofw_free_1.shp,84716,51751,478765f9,"[2025, 1, 2, 2, 55, 50]"
40,gis_osm_pofw_free_1.shx,24276,6548,f16ef3c8,"[2025, 1, 2, 2, 55, 50]"
41,gis_osm_pois_a_free_1.cpg,6,6,59b2d65b,"[2025, 1, 2, 2, 55, 20]"
42,gis_osm_pois_a_free_1.dbf,7188102,706445,f4a0d9b6,"[2025, 1, 2, 2, 55, 50]"
43,gis_osm_pois_a_free_1.prj,144,122,0bbee490,"[2025, 1, 2, 2, 55, 20]"
44,gis_osm_pois_a_free_1.shp,12261164,7178190,f56c8336,"[2025, 1, 2, 2, 55, 50]"
45,gis_osm_pois_a_free_1.shx,396676,185645,f96e28e5,"[2025, 1, 2, 2, 55, 50]"


... (23 outputs left out)

**類別前12項：bridge**

,bridge,N,ratio_of_loaded_rows
0,F,88373,0.948056
1,T,4842,0.051944


**類別前12項：tunnel**

,tunnel,N,ratio_of_loaded_rows
0,F,92375,0.990989
1,T,840,0.009011


本文件只確認資料身分及可讀性；語義／日期不足者限制用途，不強行納入正式Dataset B3。人類驗收仍待完成。

In [6]:
for item in INPUTS:ctx.file(item['path'])
print('原始容器hash重核通過；正式分析資料輸出0。')

原始容器hash重核通過；正式分析資料輸出0。
